In [1]:
import yaml
from ultralytics import YOLO

print("🏗️ Correcting YOLOv11m-CBAM Architecture Math...")

custom_yaml = """
nc: 9  
scales:
  m: [0.50, 0.50, 512] 

backbone:
  - [-1, 1, Conv, [64, 3, 2]] 
  - [-1, 1, Conv, [128, 3, 2]] 
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]] 
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]] 
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [512, 3, 2]] 
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, SPPF, [512, 5]] 
  - [-1, 2, C2PSA, [512]] 

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] 
  - [-1, 2, C3k2, [512, False]] 

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] 
  - [-1, 2, C3k2, [256, False]] 

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] 
  - [-1, 2, C3k2, [512, False]] 

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] 
  - [-1, 2, C3k2, [512, True]] 
  
  # === THE MAGIC FIX IS HERE ===
  # The 50% width multiplier means P5 outputs 256 channels!
  - [-1, 1, ULTIMATE_CBAM, [256]] 

  - [[16, 19, 23], 1, Detect, [nc]] 
"""

with open('/kaggle/working/yolov11m-cbam.yaml', 'w') as f:
    f.write(custom_yaml)

print("✅ Architecture math corrected to 256 channels!")
print("🚀 Launching 3-Epoch Dry Run on Dual GPUs!")

model = YOLO('/kaggle/working/yolov11m-cbam.yaml') 

results = model.train(
    data='/kaggle/working/fixed_data.yaml',
    epochs=3,             
    imgsz=640,
    batch=32,             
    device=[0, 1],        
    project='/kaggle/working/thesis_results', 
    name='cbam_test_run',  
    plots=False,          
    verbose=True
)
print("✅ CBAM DRY RUN FINISHED!")

ModuleNotFoundError: No module named 'ultralytics'

In [ ]:
import yaml
from ultralytics import YOLO

print("🏗️ Building the True 20M-Parameter YOLOv11m-CBAM Architecture...")

custom_yaml = """
nc: 9  
scales:
  # The true Medium scale: [depth, width, max_channels]
  m: [0.50, 1.00, 512] 

# The EXACT official YOLOv11 backbone base channels
backbone:
  - [-1, 1, Conv, [64, 3, 2]] 
  - [-1, 1, Conv, [128, 3, 2]] 
  - [-1, 2, C3k2, [256, False, 0.25]]
  - [-1, 1, Conv, [256, 3, 2]] 
  - [-1, 2, C3k2, [512, False, 0.25]]
  - [-1, 1, Conv, [512, 3, 2]] 
  - [-1, 2, C3k2, [512, True]]
  - [-1, 1, Conv, [1024, 3, 2]] 
  - [-1, 2, C3k2, [1024, True]]
  - [-1, 1, SPPF, [1024, 5]] 
  - [-1, 2, C2PSA, [1024]] 

head:
  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 6], 1, Concat, [1]] 
  - [-1, 2, C3k2, [512, False]] 

  - [-1, 1, nn.Upsample, [None, 2, "nearest"]]
  - [[-1, 4], 1, Concat, [1]] 
  - [-1, 2, C3k2, [256, False]] 

  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 13], 1, Concat, [1]] 
  - [-1, 2, C3k2, [512, False]] 

  - [-1, 1, Conv, [512, 3, 2]]
  - [[-1, 10], 1, Concat, [1]] 
  - [-1, 2, C3k2, [1024, True]] # Layer 22
  
  # === OUR ATTENTION MODULE ===
  - [-1, 1, ULTIMATE_CBAM, [512]] # Layer 23: Expects exactly 512 channels

  - [[16, 19, 23], 1, Detect, [nc]] # Layer 24
"""

with open('/kaggle/working/yolov11m-cbam.yaml', 'w') as f:
    f.write(custom_yaml)

print("🚀 LAUNCHING THE FINAL 50-EPOCH THESIS RUN!")

# 1. Load the custom architecture and transfer the COCO weights!
model = YOLO('/kaggle/working/yolov11m-cbam.yaml').load('yolo11m.pt')

# 2. Start the 50-Epoch Master Training
results = model.train(
    data='/kaggle/working/fixed_data.yaml',
    epochs=50,            
    imgsz=640,
    batch=32,             
    device=[0, 1],        
    project='/kaggle/working/thesis_results', 
    name='cbam_master_run',  
    plots=True,           
    save_period=1,        # 🛡️ BACKUP SAFEGUARD: Saves a checkpoint every single epoch!
    verbose=True
)
print("✅ IEEE ABLATION STUDY COMPLETE!")